In [1]:
# Install PySCF and GPU4PySCF (CUDA 12.x version for Colab's T4 GPU)
# cuTENSOR and CuPy are also installed for maximum tensor contraction acceleration.
!pip install pyscf gpu4pyscf-cuda12x cupy-cuda12x pyscf-dispersion geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.0/386.0 kB 8.0 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.6/51.6 MB 38.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 MB 13.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.7/159.7 MB 11.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 61.1 MB/s eta 0:00:0000:01:00:01
  Created wheel for geometric: filename=geometric-1.1-py3-none-any.whl size=402087 sha256=242635437c4888e79f5b6fdf1afbb3f82ac1d9a2f8a3273017693856b81fc486
  Stored in directory: /root/.cache/pip/wheels/df/1e/9a/763451b0dfd8db47fc02239e33cdf138cbebdbea352bb0871b
Successfully built geometric


**Only use the next cell in case multiple GPUs are used**

In [2]:
# Must be set BEFORE any gpu4pyscf imports
import gpu4pyscf.__config__ as gpu4pyscf_config
gpu4pyscf_config.num_devices = 2

# Verify both GPUs
import cupy as cp
for i in range(2):
    with cp.cuda.Device(i):
        free, total = cp.cuda.runtime.memGetInfo()
        print(f"GPU {i}: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")

/usr/local/lib/python3.12/dist-packages/gpu4pyscf/lib/cutensor.py:154: UserWarning: using cupy as the tensor contraction engine.
  warnings.warn(f'using {contract_engine} as the tensor contraction engine.')


GPU 0: 15.5 GB free / 15.6 GB total
GPU 1: 15.5 GB free / 15.6 GB total


In [3]:
import os
# Please use an xTB/other tool optimised file. Example naming is xtbopt.xyz. Please refrain from using other file formats 
# Copy the file path uploaded as a dataset in the input tab of Kaggle or system path if run locally 
xyz_filename = "/kaggle/input/datasets/pcsciprav/xtbopt-example/xtbopt.xyz"

print(f"Successfully loaded: {xyz_filename}")

Successfully loaded: /kaggle/input/datasets/pcsciprav/xtbopt-example/xtbopt.xyz


In [4]:
import cupy as cp

# Verify both GPUs are visible
n_gpus = cp.cuda.runtime.getDeviceCount()
print(f"GPUs available: {n_gpus}")
for i in range(n_gpus):
    with cp.cuda.Device(i):
        props = cp.cuda.runtime.getDeviceProperties(i)
        print(f"  GPU {i}: {props['name'].decode()}, {props['totalGlobalMem'] / 1e9:.1f} GB")

GPUs available: 2
  GPU 0: Tesla T4, 15.6 GB
  GPU 1: Tesla T4, 15.6 GB


In [5]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'  # Enable both T4s before GPU objects

import pyscf
from pyscf import gto
from gpu4pyscf.dft import rks
from pyscf.geomopt.geometric_solver import optimize
import cupy as cp

# 2. Initialize the Molecule from the uploaded xyz file
with open(xyz_filename, 'r') as f:
    lines = f.readlines()

# XYZ format: line 0 = atom count, line 1 = comment, line 2+ = atoms
n_atoms = int(lines[0].strip())
atom_lines = lines[2:2 + n_atoms]  # skip the xtb comment line
xyz_contents = ''.join(atom_lines)

mol = gto.M(
    atom=xyz_contents,
    basis='def2-SVP', # Use def2-TZVP for higher accuracy, def2-SVP is lighter and causes less OOM issues
    charge=0,
    spin=0,
    verbose=4,
    unit='angstrom'
)
mol.max_memory = 9000
print(f"\nMolecule initialized: {mol.natm} atoms, {mol.nao} atomic orbitals.")

# 3. Configure the GPU-Accelerated DFT Calculation
mf = rks.RKS(mol)
mf.xc = 'PBE-d3bj'
mf = mf.density_fit()

# Free all VRAM before optimization
cp.get_default_memory_pool().free_all_blocks()
cp.get_default_pinned_memory_pool().free_all_blocks()

# Lower memory usage per SCF cycle
mf.max_memory = 9000
mf.max_cycle = 400
mf.conv_tol = 1e-8
mf.conv_tol_grad = 1e-5
mf.diis_space = 12
mf.level_shift = 0.3
mf.damp = 0.4

# DO NOT use a fresh minao guess here
# mf.init_guess = 'minao'

# 4. RUN GEOMETRY OPTIMIZATION
cp.get_default_memory_pool().free_all_blocks()
cp.get_default_pinned_memory_pool().free_all_blocks()

print("\nStarting GPU-accelerated Geometry Optimization...")

def save_callback(envs):
    mol_envs = envs['mol']
    mol_envs.tofile('/kaggle/working/dft_optimized_checkpoint.xyz')

mol_eq = optimize(
    mf,
    maxsteps=350,
    trust=0.03,
    tmax=0.03,
    callback=save_callback,
    assert_convergence=False
)

# 5. Save the results
output_file = 'dft_optimized.xyz'
mol_eq.tofile(output_file)

print("\n" + "="*50)
print(f"Optimization complete! New coordinates saved to: {output_file}")
print("="*50)

System: uname_result(system='Linux', node='840b0d215451', release='6.6.113+', version='#1 SMP Sat Jan 17 11:20:45 UTC 2026', machine='x86_64')  Threads 4
Python 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
numpy 2.0.2  scipy 1.16.3  h5py 3.15.1
Date: Sat Mar 28 16:57:13 2026
PySCF version 2.12.1
PySCF path  /usr/local/lib/python3.12/dist-packages/pyscf/__init__.py
CUDA Environment
    CuPy 14.0.1
    CUDA Path None
    CUDA Build Version 12090
    CUDA Driver Version 13000
    CUDA Runtime Version 12090
CUDA toolkit
    cuSolver (11, 7, 3)
    cuBLAS 120804
    cuTENSOR None
Device info
    Device name b'Tesla T4'
    Device global memory 14.56 GB
    CuPy memory fraction 0.9
    Num. Devices 2
GPU4PySCF 1.6.1
GPU4PySCF path  /usr/local/lib/python3.12/dist-packages/gpu4pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 24
[INPUT] num. electrons = 168
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mol

geometric-optimize called with the following command line:
/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py -f /root/.local/share/jupyter/runtime/kernel-68592e61-7b0f-4c40-9900-428ffc50b0a8.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      *%%%%%%,  **              ********    


Geometry optimization cycle 1
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   N   3.335020  -3.025343   1.355420    0.000000  0.000000  0.000000
   N   3.866167  -1.860790   1.067908    0.000000  0.000000  0.000000
   N   3.247635  -0.894537   0.441415    0.000000  0.000000  0.000000
   N   3.715663   0.197264   0.146112   -0.000000  0.000000  0.000000
   N  -0.216286   1.256469   1.420641    0.000000  0.000000 -0.000000
   N  -0.645380   0.152965   1.705902    0.000000  0.000000  0.000000
   N  -0.015341  -0.790306   2.345917    0.000000  0.000000  0.000000
   N  -0.502489  -1.962198   2.650389    0.000000  0.000000  0.000000
   N   0.320325  -2.678915   3.257923    0.000000  0.000000 -0.000000
   N   1.483243  -2.088659   3.454880    0.000000  0.000000  0.000000
   N   1.480154  -0.915164   2.972085    0.000000  0.000000  0.000000
   N   2.384302  -0.044490   2.946058    0.000000  0.000000  0.000000
   N   6.257012  -1.173844   1.6

Step    0 : Gradient = 4.053e-02/5.926e-02 (rms/max) Energy = -1310.8944417781
Hessian Eigenvalues: 2.30001e-02 2.30002e-02 2.30004e-02 ... 8.20229e-01 9.21794e-01 9.46389e-01



Geometry optimization cycle 2
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   N   3.322809  -3.029809   1.322433   -0.012211 -0.004466 -0.032987
   N   3.883509  -1.861363   1.045032    0.017343 -0.000573 -0.022876
   N   3.223619  -0.909705   0.403302   -0.024016 -0.015168 -0.038113
   N   3.696329   0.204986   0.103699   -0.019334  0.007721 -0.042413
   N  -0.247840   1.284521   1.409327   -0.031554  0.028052 -0.011313
   N  -0.682601   0.157028   1.701153   -0.037221  0.004063 -0.004749
   N  -0.008837  -0.770702   2.355302    0.006504  0.019604  0.009385
   N  -0.527451  -1.949853   2.644403   -0.024962  0.012345 -0.005987
   N   0.311313  -2.681243   3.267082   -0.009012 -0.002328  0.009159
   N   1.499161  -2.088914   3.486236    0.015918 -0.000255  0.031356
   N   1.495484  -0.898712   2.999166    0.015330  0.016452  0.027081
   N   2.416433  -0.027815   2.988721    0.032131  0.016674  0.042663
   N   6.306278  -1.177972   1.6

...


Geometry optimization cycle 63
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   N   3.412692  -3.046054   0.878947   -0.003613  0.000082 -0.002493
   N   3.888782  -1.862846   0.614470   -0.004214  0.000656 -0.000952
   N   3.090318  -0.922798   0.019307   -0.005128  0.001772  0.002013
   N   3.582196   0.201189  -0.250824   -0.005398  0.002221  0.003110
   N  -0.126265   1.431354   1.791956   -0.001505  0.000728  0.011759
   N  -0.738939   0.430833   2.234781   -0.001593 -0.000268  0.009531
   N  -0.025616  -0.671824   2.619700   -0.002292 -0.001403  0.006314
   N  -0.637308  -1.753924   3.007170   -0.002633 -0.002295  0.003759
   N   0.136919  -2.679414   3.454573   -0.002408 -0.004070  0.000053
   N   1.448393  -2.275610   3.550832   -0.002332 -0.005271 -0.000348
   N   1.598124  -1.080801   3.162630   -0.001819 -0.004169  0.003010
   N   2.614307  -0.340582   2.939249   -0.001365 -0.004367  0.004236
   N   6.145925  -0.962882   1.

Step   62 : Displace = 1.412e-03/2.043e-03 (rms/max) Trust = 3.000e-02 (=) Grad = 1.161e-04/1.864e-04 (rms/max) E (change) = -1310.9484773858 (-6.764e-08) Quality = 0.155
Hessian Eigenvalues: 2.78024e-04 1.92768e-03 2.49323e-03 ... 7.89524e-01 8.43223e-01 8.98836e-01



Geometry optimization cycle 64
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   N   3.411832  -3.045420   0.878291   -0.000860  0.000635 -0.000656
   N   3.888381  -1.862309   0.614301   -0.000401  0.000537 -0.000169
   N   3.090140  -0.921751   0.019331   -0.000178  0.001047  0.000024
   N   3.582376   0.202085  -0.250726    0.000180  0.000895  0.000099
   N  -0.127221   1.430689   1.792892   -0.000956 -0.000665  0.000937
   N  -0.739952   0.430069   2.235425   -0.001013 -0.000764  0.000645
   N  -0.026481  -0.672447   2.620149   -0.000866 -0.000623  0.000449
   N  -0.637722  -1.754567   3.007930   -0.000414 -0.000644  0.000760
   N   0.136550  -2.680051   3.455314   -0.000369 -0.000637  0.000741
   N   1.448057  -2.276120   3.551662   -0.000336 -0.000510  0.000830
   N   1.598090  -1.081439   3.163574   -0.000034 -0.000638  0.000944
   N   2.614167  -0.341210   2.939928   -0.000140 -0.000627  0.000679
   N   6.145519  -0.963272   1.

Step   63 : Displace = 7.322e-04/1.253e-03 (rms/max) Trust = 7.061e-04 (-) Grad = 1.079e-04/2.017e-04 (rms/max) E (change) = -1310.9484777040 (-3.181e-07) Quality = 0.804
Hessian Eigenvalues: 2.78024e-04 1.92768e-03 2.49323e-03 ... 7.89524e-01 8.43223e-01 8.98836e-01
Converged! =D

    #==========================================================================#
    #| If this code has benefited your research, please support us by citing: |#
    #|                                                                        |#
    #| Wang, L.-P.; Song, C.C. (2016) "Geometry optimization made simple with |#
    #| translation and rotation coordinates", J. Chem, Phys. 144, 214108.     |#
    #| http://dx.doi.org/10.1063/1.4952956                                    |#
    #==========================================================================#
    Time elapsed since start of run_optimizer: 935.418 seconds



Optimization complete! New coordinates saved to: dft_optimized.xyz


In [6]:
import os
for f in os.listdir('/kaggle/working'):
    print(f)
import glob
print(glob.glob('/kaggle/working/*.xyz'))
print(glob.glob('/kaggle/working/**/*.xyz', recursive=True))

.virtual_documents
dft_optimized.xyz
hessian_matrix.npy
dft_optimized_checkpoint.xyz
['/kaggle/working/dft_optimized.xyz', '/kaggle/working/dft_optimized_checkpoint.xyz']
['/kaggle/working/dft_optimized.xyz', '/kaggle/working/dft_optimized_checkpoint.xyz']


In [7]:
# Cell 4: Vibrational Frequencies and Thermochemistry
from gpu4pyscf.dft import rks
from gpu4pyscf.hessian import rks as gpu_hessian
from pyscf.hessian import thermo
import cupy as cp
import numpy as np
from pyscf import gto
import numpy as np

with open('/kaggle/working/dft_optimized.xyz', 'r') as f:
    lines = f.readlines()
n_atoms = int(lines[0].strip())
atom_lines = lines[2:2 + n_atoms]
xyz_contents = ''.join(atom_lines)

mol_eq = gto.M(
    atom = xyz_contents,
    basis = '3-21g',
    charge = 0,
    spin = 0,
    verbose = 4,
    unit = 'angstrom'
)
mol_eq.max_memory = 10000

print("1. Setting up new SCF for the optimized geometry...")
mf_eq = rks.RKS(mol_eq)
mf_eq.xc = 'PBE-d3bj'
mf_eq = mf_eq.density_fit()
mf_eq.conv_tol = 1e-9
e_opt = mf_eq.kernel()

print("\n2. Calculating analytical Hessian (Frequencies) on the GPU...")
hess = gpu_hessian.Hessian(mf_eq)
hess_matrix = hess.kernel()

print("\n3. Calculating Thermochemistry (298.15 K, 1 atm)...")
freq_info = thermo.harmonic_analysis(mol_eq, hess_matrix)
thermo_data = thermo.thermo(mf_eq, freq_info['freq_au'], 298.15, 101325.0)

print("\n--- Thermochemistry Summary ---")
thermo.dump_thermo(mol_eq, thermo_data)

# MEMORY MANAGEMENT & BACKUP
print("\n4. Saving data to disk and purging VRAM...")

# Step A: Save the massive Hessian matrix to a file so we don't lose it
np.save('hessian_matrix.npy', np.asarray(hess_matrix))
print("   -> Hessian securely saved to 'hessian_matrix.npy'")

# Step B: Explicitly delete the giant objects from Python's memory
del hess
del freq_info
del thermo_data

# Step C: Force the GPU to empty the trash now that Python let go of the variables
cp.get_default_memory_pool().free_all_blocks()
print("   -> GPU VRAM successfully flushed!")

print("\n" + "="*50)
print("FREQUENCY & THERMO CALCULATION COMPLETE")
print("="*50)

System: uname_result(system='Linux', node='840b0d215451', release='6.6.113+', version='#1 SMP Sat Jan 17 11:20:45 UTC 2026', machine='x86_64')  Threads 4
Python 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
numpy 2.0.2  scipy 1.16.3  h5py 3.15.1
Date: Sat Mar 28 17:12:48 2026
PySCF version 2.12.1
PySCF path  /usr/local/lib/python3.12/dist-packages/pyscf/__init__.py
CUDA Environment
    CuPy 14.0.1
    CUDA Path None
    CUDA Build Version 12090
    CUDA Driver Version 13000
    CUDA Runtime Version 12090
CUDA toolkit
    cuSolver (11, 7, 3)
    cuBLAS 120804
    cuTENSOR None
Device info
    Device name b'Tesla T4'
    Device global memory 14.56 GB
    CuPy memory fraction 0.9
    Num. Devices 2
GPU4PySCF 1.6.1
GPU4PySCF path  /usr/local/lib/python3.12/dist-packages/gpu4pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 24
[INPUT] num. electrons = 168
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mol

In [8]:
import numpy as np
import cupy as cp

print("="*50)
print("HESSIAN / EIGENVALUE SUMMARY")
print("="*50)

H = hess_matrix
if isinstance(H, cp.ndarray):
    H = cp.asnumpy(H)

# block Hessian (natm,natm,3,3) -> full matrix
natm = H.shape[0]
Hfull = H.transpose(0,2,1,3).reshape(3*natm, 3*natm)

w = np.linalg.eigvalsh(Hfull)

print("Shape of full Hessian:", Hfull.shape)
print("Smallest 20 eigenvalues:")
print(w[:20])
print("Largest 10 eigenvalues:")
print(w[-10:])

tol = 1e-8
nneg = np.sum(w < -tol)
nzero = np.sum(np.abs(w) <= tol)
npos = np.sum(w > tol)

print("Negative eigenvalues:", nneg)
print("Near-zero eigenvalues:", nzero)
print("Positive eigenvalues:", npos)
print("="*50)
print("SOFTEST MODE ATOMIC PARTICIPATION")
print("="*50)

eigvals, eigvecs = np.linalg.eigh(Hfull)

for mode in range(min(5, len(eigvals))):
    vec = eigvecs[:, mode].reshape(natm, 3)
    atom_amp = np.linalg.norm(vec, axis=1)
    top = np.argsort(atom_amp)[::-1][:10]
    print(f"\nMode {mode}  eigenvalue = {eigvals[mode]: .8f}")
    for i in top:
        print(f"Atom {i:2d} {mol_eq.atom_symbol(i):2s}  amplitude = {atom_amp[i]:.6f}")
del hess_matrix

HESSIAN / EIGENVALUE SUMMARY
Shape of full Hessian: (72, 72)
Smallest 20 eigenvalues:
[-3.56950629e-02 -3.46109077e-02 -2.18472280e-02 -2.00304550e-02
 -1.24518530e-02 -1.14115348e-02 -9.21992179e-03 -8.85542588e-03
 -8.35671609e-03 -7.80942394e-03 -7.08782563e-03 -6.83752815e-03
 -6.89983846e-04 -1.57495280e-04 -7.46125482e-07  3.59040738e-04
  5.42398765e-04  1.60957054e-03  4.09904669e-03  4.39745015e-03]
Largest 10 eigenvalues:
[0.84000598 0.8451795  0.87235438 0.87296239 1.0229679  1.02727335
 1.11316354 1.12332092 1.30851936 1.3209646 ]
Negative eigenvalues: 15
Near-zero eigenvalues: 0
Positive eigenvalues: 57
SOFTEST MODE ATOMIC PARTICIPATION

Mode 0  eigenvalue = -0.03569506
Atom 21 N   amplitude = 0.553081
Atom 23 N   amplitude = 0.543362
Atom 20 N   amplitude = 0.372648
Atom 22 N   amplitude = 0.369966
Atom  5 N   amplitude = 0.177746
Atom  4 N   amplitude = 0.145696
Atom 11 N   amplitude = 0.099772
Atom 18 N   amplitude = 0.099396
Atom 16 N   amplitude = 0.098909
Atom 10 N  

In [9]:
# Cell 5: Excited States (TD-DFT) for UV-Vis
from gpu4pyscf.tdscf import rks as gpu_tdscf
import cupy as cp
from pyscf import gto
from gpu4pyscf.dft import rks

with open('dft_optimized_checkpoint.xyz', 'r') as f:
    lines = f.readlines()
n_atoms = int(lines[0].strip())
atom_lines = lines[2:2 + n_atoms]
xyz_contents = ''.join(atom_lines)

mol_eq = gto.M(
    atom = xyz_contents,
    basis = 'def2-svp',
    charge = 0,
    spin = 0,
    verbose = 4,
    unit = 'angstrom'
)
mol_eq.max_memory = 10000

mf_eq = rks.RKS(mol_eq)
mf_eq.xc = 'PBE-d3bj'
mf_eq = mf_eq.density_fit()
mf_eq.conv_tol = 1e-9
mf_eq.kernel()
print("Cleaning up GPU VRAM from previous calculations...")
# ==========================================
# CRITICAL VRAM CLEANUP
# Forces the GPU to release cached memory from the Hessian calc
# ==========================================
cp.get_default_memory_pool().free_all_blocks()

print("\nStarting GPU-accelerated TD-DFT...")
print("Calculating the first 5 singlet excited states...\n")

# Initialize Time-Dependent DFT using our optimized molecule's SCF object (mf_eq)
td = gpu_tdscf.TDDFT(mf_eq)
td.nstates = 5       # Number of excited states to calculate
td.singlet = True    # Look for singlet-singlet transitions

# ==========================================
# VRAM OPTIMIZATION FOR TD-DFT
# Restricts the Davidson solver's maximum memory footprint
# ==========================================
td.max_space = 15

# Run the calculation
td.kernel()

print("\n" + "="*50)
print("TD-DFT EXCITATION ENERGIES:")
print("="*50)
# This will print the transition energies (in eV), wavelengths (in nm), and oscillator strengths (intensity)
td.analyze()

System: uname_result(system='Linux', node='840b0d215451', release='6.6.113+', version='#1 SMP Sat Jan 17 11:20:45 UTC 2026', machine='x86_64')  Threads 4
Python 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
numpy 2.0.2  scipy 1.16.3  h5py 3.15.1
Date: Sat Mar 28 17:14:18 2026
PySCF version 2.12.1
PySCF path  /usr/local/lib/python3.12/dist-packages/pyscf/__init__.py
CUDA Environment
    CuPy 14.0.1
    CUDA Path None
    CUDA Build Version 12090
    CUDA Driver Version 13000
    CUDA Runtime Version 12090
CUDA toolkit
    cuSolver (11, 7, 3)
    cuBLAS 120804
    cuTENSOR None
Device info
    Device name b'Tesla T4'
    Device global memory 14.56 GB
    CuPy memory fraction 0.9
    Num. Devices 2
GPU4PySCF 1.6.1
GPU4PySCF path  /usr/local/lib/python3.12/dist-packages/gpu4pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 24
[INPUT] num. electrons = 168
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mol

<class 'gpu4pyscf.tdscf.rks.TDDFT'> does not have attributes  max_space


Excited State energies (eV)
[1.73981474 2.00267126 2.16761777 2.20383347 2.67006014]

TD-DFT EXCITATION ENERGIES:

** Singlet excitation energies and oscillator strengths **
Excited State   1:      1.73981 eV    712.63 nm  f=0.0001
      83 -> 85       -0.13430
      84 -> 85        0.68736
Excited State   2:      2.00267 eV    619.09 nm  f=0.0036
      83 -> 85        0.18239
      83 -> 86       -0.36245
      84 -> 86       -0.56904
Excited State   3:      2.16762 eV    571.98 nm  f=0.0008
      83 -> 85        0.52066
      83 -> 86       -0.29712
      84 -> 86        0.35557
Excited State   4:      2.20383 eV    562.58 nm  f=0.0020
      83 -> 85       -0.40715
      83 -> 86       -0.52514
      84 -> 85       -0.12871
      84 -> 86        0.17594
Excited State   5:      2.67006 eV    464.35 nm  f=0.0002
      83 -> 87        0.11994
      84 -> 87       -0.69392

** Transition electric dipole moments (AU) **
state          X           Y           Z        Dip. S.      Osc.
  1

In [10]:
print("="*50)
print("TDDFT SUMMARY")
print("="*50)

print("Excitation energies (eV):")
print(cp.asnumpy(td.e) * 27.211386245988 if hasattr(td.e, "get") or "cupy" in str(type(td.e)).lower() else td.e * 27.211386245988)

print("\nOscillator strengths:")
try:
    print(td.oscillator_strength())
except Exception as e:
    print("Could not get oscillator strengths directly:", e)

TDDFT SUMMARY
Excitation energies (eV):
[1.73981475 2.00267128 2.16761779 2.20383348 2.67006016]

Oscillator strengths:
[0.00010846 0.00357562 0.00078566 0.00201598 0.0002217 ]


In [11]:
import cupy as cp
from pyscf.scf import hf

print("="*50)
print("1. MOLECULAR DIPOLE MOMENT")
print("="*50)
dipole = mf_eq.dip_moment()

print("\n" + "="*50)
print("2. MULLIKEN ATOMIC CHARGES")
print("="*50)

dm = mf_eq.make_rdm1()
s = mol_eq.intor('int1e_ovlp')

# Convert GPU array to CPU NumPy array
dm = cp.asnumpy(dm)

mulliken_pop, mulliken_chg = hf.mulliken_pop(mol_eq, dm, s=s)

print("\nAtom | Charge")
print("-" * 20)
for i, charge in enumerate(mulliken_chg):
    atom_symbol = mol_eq.atom_symbol(i)
    print(f"{i:2d} {atom_symbol:2s} | {charge: .4f}")

1. MOLECULAR DIPOLE MOMENT
Dipole moment(X, Y, Z, Debye): -0.16215,  0.40827,  0.06574

2. MULLIKEN ATOMIC CHARGES
 ** Mulliken pop  **
pop of  0 N 1s            1.98793
pop of  0 N 2s            1.08602
pop of  0 N 3s            0.55958
pop of  0 N 2px           0.86615
pop of  0 N 2py           0.71416
pop of  0 N 2pz           0.74522
pop of  0 N 3px           0.38112
pop of  0 N 3py           0.19294
pop of  0 N 3pz           0.37984
pop of  0 N 3dxy          0.01283
pop of  0 N 3dyz          0.00823
pop of  0 N 3dz^2         0.00899
pop of  0 N 3dxz          0.00827
pop of  0 N 3dx2-y2       0.01112
pop of  1 N 1s            1.98762
pop of  1 N 2s            1.08287
pop of  1 N 3s            0.39907
pop of  1 N 2px           0.74998
pop of  1 N 2py           0.75957
pop of  1 N 2pz           0.81532
pop of  1 N 3px           0.45180
pop of  1 N 3py           0.28677
pop of  1 N 3pz           0.50642
pop of  1 N 3dxy          0.01285
pop of  1 N 3dyz          0.00861
pop of  1 N 3d

In [12]:
print("="*50)
print("ATOM INDEX MAP")
print("="*50)

for i in range(mol_eq.natm):
    x, y, z = mol_eq.atom_coord(i)
    print(f"{i:2d} {mol_eq.atom_symbol(i):2s}  {x: .6f} {y: .6f} {z: .6f}")

ATOM INDEX MAP
 0 N    6.447428 -5.755009  1.659729
 1 N    7.347975 -3.519255  1.160861
 2 N    5.839519 -1.741857  0.036531
 3 N    6.769710  0.381885 -0.473803
 4 N   -0.240413  2.703610  3.388076
 5 N   -1.398307  0.812712  4.224342
 6 N   -0.050043 -1.270741  4.951364
 7 N   -1.205120 -3.315652  5.684164
 8 N    0.258043 -5.064563  6.529598
 9 N    2.736431 -4.301243  6.711669
10 N    3.019953 -2.043624  5.978288
11 N    4.940060 -0.644793  5.555659
12 N    11.613348 -1.820319  3.325935
13 N    10.002570 -3.602186  3.211198
14 N    9.947269 -5.870573  3.965540
15 N    7.878807 -7.146681  3.052532
16 N    11.643696  2.600046  2.228495
17 N    10.373439  0.340590  2.756764
18 N    9.196337  0.731664  0.103756
19 N    10.313078  3.171587  0.295059
20 N    4.087546  1.760098  5.374927
21 N    5.607802  3.694740  4.370291
22 N    3.891517  4.613975  2.933818
23 N    2.263414  2.510018  3.201343


In [13]:
import numpy as np

print("="*50)
print("CHARGE SUMMARY")
print("="*50)

charges = np.array(mulliken_chg)

print("Min charge:", charges.min())
print("Max charge:", charges.max())
print("Mean charge:", charges.mean())
print("Std dev:", charges.std())

print("\nTop 10 most positive atoms")
for i in np.argsort(charges)[::-1][:10]:
    print(f"Atom {i:2d} {mol_eq.atom_symbol(i):2s}  q = {charges[i]: .6f}")

print("\nTop 10 most negative atoms")
for i in np.argsort(charges)[:10]:
    print(f"Atom {i:2d} {mol_eq.atom_symbol(i):2s}  q = {charges[i]: .6f}")

CHARGE SUMMARY
Min charge: -0.09857308517649699
Max charge: 0.0748345404466626
Mean charge: 1.5913196686293911e-15
Std dev: 0.059651462031438264

Top 10 most positive atoms
Atom  4 N   q =  0.074835
Atom  3 N   q =  0.057066
Atom  7 N   q =  0.056790
Atom 13 N   q =  0.056679
Atom 10 N   q =  0.055692
Atom 12 N   q =  0.055470
Atom  5 N   q =  0.052108
Atom 16 N   q =  0.045489
Atom 22 N   q =  0.043557
Atom  0 N   q =  0.037608

Top 10 most negative atoms
Atom 20 N   q = -0.098573
Atom 17 N   q = -0.095443
Atom 18 N   q = -0.092183
Atom 23 N   q = -0.091778
Atom  6 N   q = -0.089714
Atom  1 N   q = -0.085467
Atom 15 N   q = -0.037812
Atom  8 N   q = -0.036670
Atom  9 N   q =  0.002431
Atom 14 N   q =  0.004218


In [14]:
# Cell 7: Save the optimized coordinates
output_xyz = 'dft_optimized_final.xyz'
mol_eq.tofile(output_xyz)
print(f"Saved to /kaggle/working/{output_xyz}")
print("Download it from the Output tab on the right panel.")

Saved to /kaggle/working/dft_optimized_final.xyz
Download it from the Output tab on the right panel.
